In [ ]:
# Configure the repository root once so the same notebook works locally and on Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [ ]:
import os
import sys
import json
import torch
import numpy as np
import pandas as pd
import random

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed(42)
print("Đã cố định random seed!")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")

# 1. Nạp evaluator chuẩn
sys.path.append(str(DATA_DIR / "src/eval")) 
from evaluator import evaluate

Đã cố định random seed!
Sử dụng thiết bị: cuda


In [ ]:
with open(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/item_index.json"), 'r') as f:
    course_ids = json.load(f)
with open(str(DATA_DIR / "data/embeddings/02_lightgcn_backbone_warm/user_index_warm.json"), 'r') as f:
    warm_users = json.load(f)

assert len(course_ids) == 3769 and course_ids == sorted(course_ids), "Lỗi item_index.json"
assert warm_users == sorted(warm_users), 'user_index_warm.json phải được sắp xếp!'

item2idx = {str(c): i for i, c in enumerate(course_ids)}
user2idx = {str(u): i for i, u in enumerate(warm_users)}

g_u_warm = np.load(str(DATA_DIR / "data/embeddings/02_lightgcn_backbone_warm/g_u_warm.npy"))
g_i_warm = np.load(str(DATA_DIR / "data/embeddings/02_lightgcn_backbone_warm/g_i_warm.npy"))

assert g_u_warm.shape == (len(warm_users), 64) and g_i_warm.shape == (3769, 64), "Lỗi kích thước embedding nền chung cho warm"

In [ ]:
# 1. Load dữ liệu warm
tr_warm = pd.read_parquet(str(DATA_DIR / "data/splits/warm_scenario/train_warm.parquet"))
va_warm = pd.read_parquet(str(DATA_DIR / "data/splits/warm_scenario/val_warm.parquet"))
te_warm = pd.read_parquet(str(DATA_DIR / "data/splits/warm_scenario/test_warm.parquet"))

In [5]:
# 3. Xây dựng target_warm và hist_warm cho tập test warm
target_warm = {}
for u, group in te_warm.groupby('user_id'):
  u_str = str(u)
  if u_str in user2idx:
    u_idx = user2idx[u_str]
    item_idxs = [
        item2idx[str(i)] for i in group['course_id'] if str(i) in item2idx
    ]
    if len(item_idxs) > 0:
      target_warm[u_idx] = item_idxs

# Che lịch sử học tập khi chấm test (train_warm + val_warm)
hist_warm_df = pd.concat([tr_warm, va_warm])
hist_warm = {}
for u, group in hist_warm_df.groupby('user_id'):
  u_str = str(u)
  if u_str in user2idx:
    u_idx = user2idx[user2idx == u_str] if False else user2idx[u_str]
    item_idxs = [
        item2idx[str(i)] for i in group['course_id'] if str(i) in item2idx
    ]
    if len(item_idxs) > 0:
      hist_warm[u_idx] = item_idxs

# Chuyển embedding sang Torch tensor trên Device
u_emb_warm_t = torch.tensor(g_u_warm, dtype=torch.float32, device=DEVICE)
i_emb_warm_t = torch.tensor(g_i_warm, dtype=torch.float32, device=DEVICE)

def score_fn_warm(users_tensor):
  u_idx = users_tensor.to(DEVICE)
  return torch.matmul(u_emb_warm_t[u_idx], i_emb_warm_t.t())

In [6]:
cand_bool_mask_warm = np.ones(3769, dtype=bool)
SEEDS = [42, 2023, 2024, 1007, 999]
results_warm = []

print("\n--- Đánh giá LightGCN thuần ID trên Kịch bản Warm với 5 seed ---")
for s in SEEDS:
    res = evaluate(score_fn_warm, target_warm, hist_warm, ks=(5, 10, 20), cand=cand_bool_mask_warm, bs=1024, tie_seed=s)
    results_warm.append(res)

print('\n--- Kết quả Đánh giá LightGCN Thuần ID trên Warm-Test ---')
for k in results_warm[0].keys():
    seed_means = np.array([res[k].mean() for res in results_warm])
    mean_value = seed_means.mean()
    std_value = seed_means.std(ddof=1)  
    print(f"{k}: {mean_value:.4f} ± {std_value:.4f}")


--- Đánh giá LightGCN thuần ID trên Kịch bản Warm với 5 seed ---

--- Kết quả Đánh giá LightGCN Thuần ID trên Warm-Test ---
Recall@5: 0.0710 ± 0.0000
Recall@10: 0.1043 ± 0.0000
Recall@20: 0.1463 ± 0.0000
NDCG@5: 0.0611 ± 0.0000
NDCG@10: 0.0712 ± 0.0000
NDCG@20: 0.0854 ± 0.0000
HR@5: 0.1583 ± 0.0000
HR@10: 0.2415 ± 0.0000
HR@20: 0.3485 ± 0.0000
